# Exercise 2: Mobile Device Usage Profiling
**Student Name:** Solis  
**Student ID:** `TUPM-26-2631`  

### Mission
Develop a file-based profiling program that filters device records using student-specific criteria and exports a verified usage summary.

### Data Generation Rules
- **Operating System:** Android if SEED_DIGIT is even; iOS if odd -> **iOS** (SEED_DIGIT = 1 is odd)
- **User Behavior Class:** `(SEED_DIGIT % 5) + 1` = `(1 % 5) + 1` = **Class 2**
- **Exported Fields:** `User ID`, `Device Model`, `Operating System`, `Battery Drain (mAh/day)`, `Data Usage (MB/day)`


### Step 1: Student Information & Parameter Derivation


In [1]:
LAST_NAME = "Solis"
STUDENT_ID = "TUPM-26-2631"
SEED_DIGIT = int(STUDENT_ID[-1])  # 1
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 20

TARGET_OS = "Android" if (SEED_DIGIT % 2 == 0) else "iOS"
TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1  # 2

EXPORTED_FIELDS = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)",
]

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("Assigned Operating System:", TARGET_OS, "(SEED_DIGIT is odd -> iOS)")
print(f"Assigned User Behavior Class: Class {TARGET_BEHAVIOR_CLASS} ((1 % 5) + 1)")
print("Exported Fields:", EXPORTED_FIELDS)


Student: Solis
Student ID: TUPM-26-2631
Seed Digit: 1
Assigned Operating System: iOS (SEED_DIGIT is odd -> iOS)
Assigned User Behavior Class: Class 2 ((1 % 5) + 1)
Exported Fields: ['User ID', 'Device Model', 'Operating System', 'Battery Drain (mAh/day)', 'Data Usage (MB/day)']


### Step 2: Directory & File Path Setup


In [2]:
import os
import csv
import json
import shutil
from pathlib import Path

BASE_DIR = Path(".")
DATASET_PATH = BASE_DIR / "datasets" / "user_behavior_dataset.csv"
if not DATASET_PATH.exists():
    DATASET_PATH = BASE_DIR.parent / "datasets" / "user_behavior_dataset.csv"

OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

EXPORTED_CSV_PATH = OUTPUT_DIR / "user_behavior_ios_class2.csv"
JSON_SUMMARY_PATH = OUTPUT_DIR / "user_behavior_ios_class2_summary.json"
BACKUP_CSV_PATH = BACKUP_DIR / "user_behavior_ios_class2_backup.csv"

print("Dataset Path:", DATASET_PATH.name)
print("Dataset exists:", DATASET_PATH.exists())
print("Output Directory exists:", OUTPUT_DIR.exists())
print("Backup Directory exists:", BACKUP_DIR.exists())


Dataset Path: user_behavior_dataset.csv
Dataset exists: True
Output Directory exists: True
Backup Directory exists: True


### Step 3: Read Dataset & Filter Matching Operating System (iOS)


In [3]:
total_records = 0
os_matched_records = []

with open(DATASET_PATH, mode="r", encoding="utf-8", errors="ignore") as f:
    reader = csv.DictReader(f)
    print("Dataset Columns:", reader.fieldnames)
    for row in reader:
        total_records += 1
        os_val = (row.get("Operating System") or "").strip()
        if os_val.lower() == TARGET_OS.lower():
            os_matched_records.append(row)

records_matching_os = len(os_matched_records)
print(f"Total records in dataset: {total_records}")
print(f"Records matching operating system '{TARGET_OS}': {records_matching_os}")


Dataset Columns: ['User ID', 'Device Model', 'Operating System', 'App Usage Time (min/day)', 'Screen On Time (hours/day)', 'Battery Drain (mAh/day)', 'Number of Apps Installed', 'Data Usage (MB/day)', 'Age', 'Gender', 'User Behavior Class']
Total records in dataset: 700
Records matching operating system 'iOS': 146


### Step 4: Filter Users Belonging to Target Behavior Class (Class 2)


In [4]:
selected_users = []
for row in os_matched_records:
    behavior_class_val = (row.get("User Behavior Class") or "").strip()
    if behavior_class_val == str(TARGET_BEHAVIOR_CLASS):
        selected_users.append(row)

records_in_target_class = len(selected_users)
print(f"Target Behavior Class: Class {TARGET_BEHAVIOR_CLASS}")
print(f"Records in target behavior class: {records_in_target_class}")
print("\nSample selected user record:")
print({f: selected_users[0][f] for f in EXPORTED_FIELDS})


Target Behavior Class: Class 2
Records in target behavior class: 32

Sample selected user record:
{'User ID': '37', 'Device Model': 'iPhone 12', 'Operating System': 'iOS', 'Battery Drain (mAh/day)': '755', 'Data Usage (MB/day)': '573'}


### Step 5: Calculate Average Battery Drain & Mobile Data Usage


In [5]:
battery_drains = []
data_usages = []

for row in selected_users:
    try:
        battery_drains.append(float(row.get("Battery Drain (mAh/day)", 0)))
    except ValueError:
        pass
    try:
        data_usages.append(float(row.get("Data Usage (MB/day)", 0)))
    except ValueError:
        pass

avg_battery = sum(battery_drains) / len(battery_drains) if battery_drains else 0.0
avg_data = sum(data_usages) / len(data_usages) if data_usages else 0.0

print(f"Average Battery Drain: {avg_battery:.2f} mAh/day")
print(f"Average Mobile Data Usage: {avg_data:.2f} MB/day")


Average Battery Drain: 872.03 mAh/day
Average Mobile Data Usage: 451.34 MB/day


### Step 6: Export Selected Users and Specified Fields to CSV


In [6]:
export_data = [
    {field: row.get(field, "").strip() for field in EXPORTED_FIELDS}
    for row in selected_users
]

with open(EXPORTED_CSV_PATH, mode="w", encoding="utf-8", newline="") as cf:
    writer = csv.DictWriter(cf, fieldnames=EXPORTED_FIELDS)
    writer.writeheader()
    writer.writerows(export_data)

with open(EXPORTED_CSV_PATH, mode="r", encoding="utf-8") as cf:
    exported_row_count = sum(1 for _ in csv.DictReader(cf))

print(f"Exported CSV: {EXPORTED_CSV_PATH.name}")
print(f"Exported row count: {exported_row_count}")


Exported CSV: user_behavior_ios_class2.csv
Exported row count: 32


### Step 7: Write JSON Summary & Count Verification


In [7]:
count_verified = (exported_row_count == records_in_target_class)

summary_data = {
    "assessment_data": {
        "student_name": LAST_NAME,
        "student_id": STUDENT_ID,
        "seed_digit": SEED_DIGIT,
        "id_sum": ID_SUM,
        "operating_system_and_behavior_class": f"{TARGET_OS}, Class {TARGET_BEHAVIOR_CLASS}",
        "total_records_in_dataset": total_records,
        "records_matching_operating_system": records_matching_os,
        "records_in_target_behavior_class": records_in_target_class,
        "average_battery_drain_mah_day": round(avg_battery, 2),
        "average_data_usage_mb_day": round(avg_data, 2),
        "count_verification": {
            "selected_record_count": records_in_target_class,
            "exported_row_count": exported_row_count,
            "verified": count_verified
        },
        "exported_csv_file": EXPORTED_CSV_PATH.name,
        "json_summary_file": JSON_SUMMARY_PATH.name
    }
}

with open(JSON_SUMMARY_PATH, mode="w", encoding="utf-8") as jf:
    json.dump(summary_data, jf, indent=4)

print("Count Verification:")
print(f"  Selected records: {records_in_target_class}")
print(f"  Exported rows:    {exported_row_count}")
print(f"  Verification:     {'PASS (VERIFIED)' if count_verified else 'FAIL'}")
print(f"JSON summary saved: {JSON_SUMMARY_PATH.name}")


Count Verification:
  Selected records: 32
  Exported rows:    32
  Verification:     PASS (VERIFIED)
JSON summary saved: user_behavior_ios_class2_summary.json


### Step 8: Create Backup of Exported CSV & Report File Size


In [8]:
shutil.copy2(EXPORTED_CSV_PATH, BACKUP_CSV_PATH)

backup_exists = BACKUP_CSV_PATH.exists()
backup_file_size = BACKUP_CSV_PATH.stat().st_size if backup_exists else 0

summary_data["assessment_data"]["backup_file_and_size"] = {
    "file_name": BACKUP_CSV_PATH.name,
    "size_bytes": backup_file_size,
    "verified": backup_exists
}

with open(JSON_SUMMARY_PATH, mode="w", encoding="utf-8") as jf:
    json.dump(summary_data, jf, indent=4)

print("Backup File:", BACKUP_CSV_PATH.name)
print(f"Backup File Size: {backup_file_size} bytes")
print("Backup Exists:", backup_exists)


Backup File: user_behavior_ios_class2_backup.csv
Backup File Size: 954 bytes
Backup Exists: True


### Step 9: Assessment Data Table & Final Verification


In [9]:
print("=" * 70)
print(f"{'Item':<40} | {'Result'}")
print("-" * 70)
print(f"{'Operating system and behavior class':<40} | {TARGET_OS}, Class {TARGET_BEHAVIOR_CLASS}")
print(f"{'Total records in dataset':<40} | {total_records}")
print(f"{'Records matching operating system':<40} | {records_matching_os}")
print(f"{'Records in target behavior class':<40} | {records_in_target_class}")
print(f"{'Average battery drain (mAh/day)':<40} | {avg_battery:.2f} mAh/day")
print(f"{'Average data usage (MB/day)':<40} | {avg_data:.2f} MB/day")
print(f"{'Count verification':<40} | VERIFIED ({exported_row_count} == {records_in_target_class})")
print(f"{'Exported CSV file':<40} | {EXPORTED_CSV_PATH.name}")
print(f"{'JSON summary file':<40} | {JSON_SUMMARY_PATH.name}")
print(f"{'Backup file and size':<40} | {BACKUP_CSV_PATH.name} ({backup_file_size} bytes)")
print("=" * 70)
print("\nFinal Output: Exercise 2 completed successfully!")


Item                                     | Result
----------------------------------------------------------------------
Operating system and behavior class      | iOS, Class 2
Total records in dataset                 | 700
Records matching operating system        | 146
Records in target behavior class         | 32
Average battery drain (mAh/day)          | 872.03 mAh/day
Average data usage (MB/day)              | 451.34 MB/day
Count verification                       | VERIFIED (32 == 32)
Exported CSV file                        | user_behavior_ios_class2.csv
JSON summary file                        | user_behavior_ios_class2_summary.json
Backup file and size                     | user_behavior_ios_class2_backup.csv (954 bytes)

Final Output: Exercise 2 completed successfully!
